# Thermal PV Panel Binary Classifier — MobileNetV2 (Kaggle)

**Project:** UAV-Based Photovoltaic Panel Inspection  
**Task:** Binary classification — `no_anomaly` vs `anomaly`  
**Model:** MobileNetV2 (width=1.0, ImageNet pretrained)  
**Input:** 224×224×3 RGB (thermal colourmap preserved)  
**Export:** TFLite float16 for Pi 5 deployment  

**Dataset:** Merged PVF-10 + PVMD (ImageFolder layout)  
```
dataset/
  train/
    no_anomaly/
    anomaly/
  val/
    no_anomaly/
    anomaly/
  test/
    no_anomaly/
    anomaly/
```

## 0 — Environment & Reproducibility

In [ ]:
import os, random, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, mixed_precision
from tensorflow.keras.applications import MobileNetV2

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── GPU check ────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__} | GPUs: {[g.name for g in gpus]}')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('Memory growth enabled')

## 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjust DATA_ROOT to point to your Kaggle dataset input path
DATA_ROOT   = Path('/kaggle/input/thermal-pv-dataset')   # <-- update if needed
TRAIN_DIR   = DATA_ROOT / 'train'
VAL_DIR     = DATA_ROOT / 'val'
TEST_DIR    = DATA_ROOT / 'test'
OUTPUT_DIR  = Path('/kaggle/working')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 224          # MobileNetV2 canonical resolution
BATCH_SIZE  = 32
NUM_CLASSES = 1            # binary → sigmoid output
DROPOUT     = 0.3

# Phase 1 — head only
LR_HEAD     = 1e-3
EPOCHS_HEAD = 10

# Phase 2 — unfreeze top-1/3 of base
LR_FINE1    = 3e-4
EPOCHS_FINE1= 10

# Phase 3 — unfreeze full base
LR_FINE2    = 5e-5
EPOCHS_FINE2= 10

CLASSES     = ['anomaly', 'no_anomaly']   # sorted → Keras ImageFolder order

print('Config OK')
print(f'  Train : {TRAIN_DIR}')
print(f'  Val   : {VAL_DIR}')
print(f'  Test  : {TEST_DIR}')

## 2 — Dataset Loading & Class Weights

In [ ]:
def make_dataset(directory, training=False, batch_size=BATCH_SIZE):
    """Load an ImageFolder-style directory with optional augmentation."""
    ds = keras.utils.image_dataset_from_directory(
        directory,
        labels='inferred',
        label_mode='binary',          # 0/1 for sigmoid
        class_names=CLASSES,
        color_mode='rgb',
        batch_size=batch_size,
        image_size=(IMG_SIZE, IMG_SIZE),
        shuffle=training,
        seed=SEED,
        interpolation='bilinear',
    )
    return ds

train_ds_raw = make_dataset(TRAIN_DIR, training=True)
val_ds_raw   = make_dataset(VAL_DIR,   training=False)
test_ds_raw  = make_dataset(TEST_DIR,  training=False)

print('Class index mapping:', train_ds_raw.class_names)

# ── Class distribution ────────────────────────────────────────────────────────
def count_samples(directory):
    counts = {}
    for cls in CLASSES:
        p = Path(directory) / cls
        counts[cls] = len(list(p.glob('*.*'))) if p.exists() else 0
    return counts

train_counts = count_samples(TRAIN_DIR)
val_counts   = count_samples(VAL_DIR)
test_counts  = count_samples(TEST_DIR)

print('\nSample counts:')
for split, counts in [('Train', train_counts), ('Val', val_counts), ('Test', test_counts)]:
    total = sum(counts.values())
    print(f'  {split}: {counts}  → total {total}')

# ── Class weights (handles imbalance) ─────────────────────────────────────────
# anomaly=0, no_anomaly=1  (Keras sorted order)
n_anomaly    = train_counts.get('anomaly', 1)
n_no_anomaly = train_counts.get('no_anomaly', 1)
total_train  = n_anomaly + n_no_anomaly

# sklearn-style balanced weights
y_train_flat = ([0] * n_anomaly) + ([1] * n_no_anomaly)
cw = compute_class_weight('balanced', classes=np.array([0, 1]),
                           y=np.array(y_train_flat))
class_weight = {0: cw[0], 1: cw[1]}
print(f'\nClass weights: {class_weight}')

## 3 — Augmentation Pipeline

Two-stage strategy:
1. **Geometric** — flip, rotation, zoom (invariant to panel orientation)
2. **Colormap jitter** — hue/saturation shifts to bridge the PVF-10 vs PVMD colormap domain gap (iron vs rainbow palettes)

In [ ]:
# ── Preprocessing: scale to [-1, 1] (MobileNetV2 convention) ─────────────────
preprocess = keras.applications.mobilenet_v2.preprocess_input  # maps [0,255]→[-1,1]

# ── Keras augmentation layers (applied on GPU, inside tf.data) ───────────────
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),           # ±15 % of 360°
    layers.RandomZoom(0.10),               # ±10 %
    layers.RandomTranslation(0.05, 0.05),  # slight shifts
    # Colormap-aware jitter: hue shift bridges iron↔rainbow domain gap
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

# RandomHue is not built-in in older TF; use tf.image directly:
@tf.function
def augment_hue(image, label):
    """Slight hue rotation to simulate different thermal colourmap palettes."""
    image = tf.cast(image, tf.float32)
    # Random hue delta in [-0.08, 0.08] — enough to shift iron→rainbow, not so
    # much that hot/cold polarity inverts
    image = tf.image.random_hue(image / 255.0, max_delta=0.08) * 255.0
    return tf.clip_by_value(image, 0.0, 255.0), label

def build_pipeline(ds, training=False):
    AUTOTUNE = tf.data.AUTOTUNE
    if training:
        ds = ds.map(augment_hue, num_parallel_calls=AUTOTUNE)
        ds = ds.map(lambda x, y: (augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    # Preprocess to [-1, 1]
    ds = ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = build_pipeline(train_ds_raw, training=True)
val_ds   = build_pipeline(val_ds_raw,   training=False)
test_ds  = build_pipeline(test_ds_raw,  training=False)

print('Pipelines built.')

# ── Visualise one augmented batch ─────────────────────────────────────────────
sample_images, sample_labels = next(iter(train_ds_raw.take(1)))
aug_images, _ = augment_hue(sample_images[:8], sample_labels[:8])
aug_images = augmentation(aug_images, training=True)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(8):
    axes[0, i].imshow(sample_images[i].numpy().astype('uint8'))
    axes[0, i].set_title(CLASSES[int(sample_labels[i])], fontsize=7)
    axes[0, i].axis('off')
    axes[1, i].imshow(tf.clip_by_value(aug_images[i], 0, 255).numpy().astype('uint8'))
    axes[1, i].set_title('augmented', fontsize=7)
    axes[1, i].axis('off')
fig.suptitle('Top: original  |  Bottom: augmented', fontsize=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'augmentation_preview.png', dpi=120)
plt.show()

## 4 — Model Architecture

In [ ]:
def build_model(trainable_base=False):
    base = MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        alpha=1.0,           # full-width model
        include_top=False,
        weights='imagenet',
    )
    base.trainable = trainable_base

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)          # BN layers stay in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(DROPOUT)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)   # binary

    model = keras.Model(inputs, outputs)
    return model, base

model, base_model = build_model(trainable_base=False)
model.summary(line_length=90)
print(f'\nTrainable params: {model.trainable_variables.__len__()} tensors')

## 5 — Training

Three-phase progressive unfreezing:

| Phase | Layers trained | LR |
|-------|----------------|----|
| 1 | Head only | 1e-3 |
| 2 | Head + top 1/3 of base | 3e-4 |
| 3 | Full network | 5e-5 |

In [ ]:
def get_callbacks(phase_name):
    return [
        callbacks.ModelCheckpoint(
            str(OUTPUT_DIR / f'best_{phase_name}.keras'),
            monitor='val_auc', mode='max',
            save_best_only=True, verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_auc', mode='max',
            patience=5, restore_best_weights=True, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-7, verbose=1
        ),
    ]

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=[
            keras.metrics.BinaryAccuracy(name='accuracy'),
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ]
    )

history_all = {}

In [ ]:
# ── Phase 1: Train head only ──────────────────────────────────────────────────
print('=' * 60)
print('PHASE 1 — Head only')
print('=' * 60)

base_model.trainable = False
compile_model(model, LR_HEAD)

h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=get_callbacks('phase1'),
    verbose=1,
)
history_all['phase1'] = h1.history

In [ ]:
# ── Phase 2: Unfreeze top 1/3 of base ────────────────────────────────────────
print('=' * 60)
print('PHASE 2 — Top 1/3 of base unfrozen')
print('=' * 60)

base_model.trainable = True
n_layers = len(base_model.layers)
freeze_until = int(n_layers * 2 / 3)   # freeze bottom 2/3
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False
# Keep BN layers frozen to preserve ImageNet statistics
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(f'  Base layers: {n_layers} | frozen: {freeze_until} | trainable: {n_layers-freeze_until}')
compile_model(model, LR_FINE1)

h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE1,
    class_weight=class_weight,
    callbacks=get_callbacks('phase2'),
    verbose=1,
)
history_all['phase2'] = h2.history

In [ ]:
# ── Phase 3: Unfreeze full base ───────────────────────────────────────────────
print('=' * 60)
print('PHASE 3 — Full network fine-tuned')
print('=' * 60)

for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False   # keep BN frozen throughout
    else:
        layer.trainable = True

compile_model(model, LR_FINE2)

h3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE2,
    class_weight=class_weight,
    callbacks=get_callbacks('phase3'),
    verbose=1,
)
history_all['phase3'] = h3.history

## 6 — Training Curves

In [ ]:
def plot_history(history_all):
    metrics = ['loss', 'accuracy', 'auc']
    phase_colors = {'phase1': '#4C72B0', 'phase2': '#DD8452', 'phase3': '#55A868'}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    offset = 0
    for phase, hist in history_all.items():
        n = len(hist['loss'])
        epochs = range(offset + 1, offset + n + 1)
        color  = phase_colors[phase]
        for ax, metric in zip(axes, metrics):
            ax.plot(epochs, hist[metric],       color=color,  lw=2,   label=f'{phase} train')
            ax.plot(epochs, hist[f'val_{metric}'], color=color, lw=2, ls='--', label=f'{phase} val')
        offset += n

    # Vertical lines at phase boundaries
    boundaries = [EPOCHS_HEAD, EPOCHS_HEAD + EPOCHS_FINE1]
    for b, label in zip(boundaries, ['Phase 2 start', 'Phase 3 start']):
        for ax in axes:
            ax.axvline(b, color='grey', ls=':', lw=1, label=label)

    for ax, metric in zip(axes, metrics):
        ax.set_title(metric.capitalize())
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=6)
        ax.grid(alpha=0.3)

    plt.suptitle('Training curves — MobileNetV2 binary thermal classifier', y=1.02)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_all)

## 7 — Evaluation on Test Set

In [ ]:
# Load best checkpoint from Phase 3 (or whichever achieved best val_auc)
best_ckpt = OUTPUT_DIR / 'best_phase3.keras'
if not best_ckpt.exists():
    best_ckpt = OUTPUT_DIR / 'best_phase2.keras'
model = keras.models.load_model(str(best_ckpt))
print(f'Loaded checkpoint: {best_ckpt}')

# ── Collect predictions ───────────────────────────────────────────────────────
y_true, y_prob = [], []
for images, labels in test_ds:
    probs = model.predict(images, verbose=0).flatten()
    y_true.extend(labels.numpy().astype(int).flatten())
    y_prob.extend(probs)

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

# ── Metrics ───────────────────────────────────────────────────────────────────
print('\n── Classification Report ──')
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

auc = roc_auc_score(y_true, y_prob)
print(f'ROC-AUC: {auc:.4f}')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True label')
axes[0].set_xlabel('Predicted label')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)')
axes[1].set_ylabel('True label')
axes[1].set_xlabel('Predicted label')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

# ── ROC curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, lw=2, label=f'MobileNetV2 (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Binary Thermal Classifier')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=150)
plt.show()

## 8 — TFLite Float16 Export (Pi 5)

In [ ]:
# ── Save full Keras model first ───────────────────────────────────────────────
keras_path = str(OUTPUT_DIR / 'thermal_binary_mobilenetv2.keras')
model.save(keras_path)
print(f'Keras model saved: {keras_path}')

# ── Convert to TFLite float16 ─────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
# Keep input/output as float32 for easy inference on Pi 5
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

tflite_path = OUTPUT_DIR / 'thermal_binary_mobilenetv2_f16.tflite'
tflite_path.write_bytes(tflite_model)
print(f'TFLite model saved: {tflite_path}')
print(f'  Size: {tflite_path.stat().st_size / 1024 / 1024:.2f} MB')

# ── Verify TFLite model ───────────────────────────────────────────────────────
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
inp  = interpreter.get_input_details()
outp = interpreter.get_output_details()
print(f'\nTFLite input  : shape={inp[0]["shape"]}, dtype={inp[0]["dtype"]}')
print(f'TFLite output : shape={outp[0]["shape"]}, dtype={outp[0]["dtype"]}')

In [ ]:
# ── Benchmark TFLite on a batch of test images ────────────────────────────────
import time

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
inp_idx  = interpreter.get_input_details()[0]['index']
outp_idx = interpreter.get_output_details()[0]['index']

# Single-image inference time
sample_batch, _ = next(iter(test_ds.take(1)))
single_img = sample_batch[0:1].numpy().astype(np.float32)

times = []
for _ in range(50):
    t0 = time.perf_counter()
    interpreter.set_tensor(inp_idx, single_img)
    interpreter.invoke()
    _ = interpreter.get_tensor(outp_idx)
    times.append((time.perf_counter() - t0) * 1000)

print(f'TFLite inference time (50 runs, single image):')
print(f'  Mean : {np.mean(times):.1f} ms')
print(f'  Std  : {np.std(times):.1f} ms')
print(f'  Min  : {np.min(times):.1f} ms')
print(f'  Max  : {np.max(times):.1f} ms')
print('\n(Kaggle T4 GPU times; expect ~5-8× slower on Pi 5 CPU)')

## 9 — Save Metadata & Outputs

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

metadata = {
    'model'        : 'MobileNetV2',
    'alpha'        : 1.0,
    'input_size'   : [IMG_SIZE, IMG_SIZE, 3],
    'classes'      : CLASSES,
    'class_weight' : {str(k): float(v) for k, v in class_weight.items()},
    'test_metrics' : {
        'accuracy' : float(accuracy_score(y_true, y_pred)),
        'macro_f1' : float(f1_score(y_true, y_pred, average='macro')),
        'precision': float(precision_score(y_true, y_pred, average='macro')),
        'recall'   : float(recall_score(y_true, y_pred, average='macro')),
        'roc_auc'  : float(auc),
    },
    'tflite_path'  : 'thermal_binary_mobilenetv2_f16.tflite',
    'preprocessing': 'mobilenet_v2.preprocess_input  # [0,255]→[-1,1]',
}

meta_path = OUTPUT_DIR / 'model_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved metadata:')
print(json.dumps(metadata, indent=2))

print('\n── Output files ──')
for p in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {p.name:50s}  {p.stat().st_size/1024:.1f} KB')

## 10 — Pi 5 Inference Snippet

Copy `thermal_binary_mobilenetv2_f16.tflite` to the Pi 5 and use this snippet inside the main pipeline:

In [ ]:
PI5_INFERENCE_SNIPPET = '''
# ── Pi 5 inference — drop-in for existing ONNX classifier call ───────────────
import numpy as np
import cv2
from tflite_runtime.interpreter import Interpreter   # pip install tflite-runtime

IMG_SIZE = 224
THRESHOLD = 0.5

# Load once at startup
interpreter = Interpreter(model_path="thermal_binary_mobilenetv2_f16.tflite")
interpreter.allocate_tensors()
inp_idx  = interpreter.get_input_details()[0]["index"]
outp_idx = interpreter.get_output_details()[0]["index"]

def preprocess_thermal(bgr_crop):
    """BGR crop from TC002C stream → model input tensor."""
    rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    # MobileNetV2 preprocessing: [0,255] → [-1, 1]
    tensor = (resized.astype(np.float32) / 127.5) - 1.0
    return np.expand_dims(tensor, axis=0)   # (1, 224, 224, 3)

def classify_thermal(bgr_crop):
    """Returns (label_str, confidence_float)."""
    tensor = preprocess_thermal(bgr_crop)
    interpreter.set_tensor(inp_idx, tensor)
    interpreter.invoke()
    prob = float(interpreter.get_tensor(outp_idx)[0][0])
    label = "no_anomaly" if prob >= THRESHOLD else "anomaly"
    conf  = prob if prob >= THRESHOLD else 1.0 - prob
    return label, conf
'''

snippet_path = OUTPUT_DIR / 'pi5_inference_snippet.py'
snippet_path.write_text(PI5_INFERENCE_SNIPPET)
print('Pi 5 snippet saved.')
print(PI5_INFERENCE_SNIPPET)